# FlowEdit CFG++ No-First-Term Same-NFE Experiment

Goal: test the second meeting idea: remove the first anchor term from the CFG++-style update, then apply the remaining midpoint contrast correction to FlowEdit under the same estimated NFE.

For FlowEdit, define the contrastive editing direction:

$$z_t^{src}=(1-t)x_{src}+t\epsilon$$

$$z_t^{tar}=z_t^{edit}+z_t^{src}-x_{src}$$

$$G_t^{FE}=v_{tar}(z_t^{tar},t)-v_{src}(z_t^{src},t)$$

Predict source and target midpoints separately:

$$\tilde z_{mid}^{tar}=z_t^{tar}+\frac{h}{2}v_{tar}(z_t^{tar},t)$$

$$\tilde z_{mid}^{src}=z_t^{src}+\frac{h}{2}v_{src}(z_t^{src},t)$$

Then compute the midpoint contrast:

$$G_{mid}^{FE}=v_{tar}(\tilde z_{mid}^{tar},t_{mid})-v_{src}(\tilde z_{mid}^{src},t_{mid})$$

Remove the CFG++ first term and keep only the scheduled midpoint correction:

$$\hat G_t^{FE}=\alpha(t)wG_{mid}^{FE},\quad \alpha(t)=\lambda(1-t)^\gamma$$

The edit latent is updated by:

$$z_{next}^{edit}=z_t^{edit}+h\hat G_t^{FE}$$

This is intentionally different from weighted interpolation:

$$G_t^{FE}+\alpha(t)(wG_{mid}^{FE}-G_t^{FE})$$

Evaluation focus: same NFE, CLIP target alignment, DINO source preservation, runtime, and a lightweight saturation/clipping artifact proxy.


In [ ]:
# 1) Configuration
# Do not hard-code tokens in this notebook. Paste one only at runtime if SD3 access requires it.
import os
from getpass import getpass

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "codex/flowedit-remove-cfgpp-first-term"
WORKDIR = "/content/FlowEdit"

EXP_YAML = "SD3_cfgpp_no_first_term_same_nfe.yaml"
DATASET_YAML = "edits_midpoint_eval.yaml"

METRICS_DIR = "outputs/metrics"
QUALITY_PER_SAMPLE_CSV = f"{METRICS_DIR}/cfgpp_no_first_term_clip_dino_per_sample.csv"
QUALITY_SUMMARY_CSV = f"{METRICS_DIR}/cfgpp_no_first_term_clip_dino_summary.csv"
ARTIFACT_PER_SAMPLE_CSV = f"{METRICS_DIR}/cfgpp_no_first_term_artifact_per_sample.csv"
ARTIFACT_SUMMARY_CSV = f"{METRICS_DIR}/cfgpp_no_first_term_artifact_summary.csv"
COMPARISON_CSV = f"{METRICS_DIR}/cfgpp_no_first_term_comparison_table.csv"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: ")

print("Experiment YAML:", EXP_YAML)
print("Dataset YAML:", DATASET_YAML)
print("Target branch:", BRANCH)


In [ ]:
# 2) Clone repository, install dependencies, and switch to workspace
import os
import subprocess
from getpass import getpass
from pathlib import Path

# Keep Colab's preinstalled numpy/pandas/sklearn/protobuf stack intact.
# Reinstalling those core binary packages can fail or require a runtime restart.

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError("Could not checkout the branch. Push this branch first, or upload the notebook into the checked-out repo.")
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=False)

subprocess.run([
    "pip", "install", "-q", "--upgrade",
    "plotly==5.24.1",
    "diffusers>=0.31.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "safetensors",
    "sentencepiece",
    "einops",
    "pyyaml",
    "huggingface_hub",
], check=True)

import numpy as np
import pandas as pd
import sklearn
import PIL
import google.protobuf
import plotly
import diffusers
import transformers
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("Pillow", PIL.__version__)
print("protobuf", google.protobuf.__version__)
print("plotly", plotly.__version__)
print("diffusers", diffusers.__version__)
print("transformers", transformers.__version__)

HF_TOKEN = globals().get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token, leave blank if not needed: " )

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face for model download.")
else:
    print("No HF token provided. Public downloads only.")


In [ ]:
# 3) Inspect the same-NFE experiment configuration
from pathlib import Path
import yaml
import pandas as pd

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)

two_call_edit_solvers = {
    "midpoint",
    "flowedit_cfgpp_no_first_term",
    "flowedit_cfgpp_no_first",
    "flowedit_remove_cfgpp_first_term",
}

rows = []
for item in exp:
    edit_steps = max(min(item["n_max"], item["T_steps"]) - max(item["n_min"], 0), 0)
    calls_per_edit_step = 2 if item["solver_type"] in two_call_edit_solvers else 1
    estimated_nfe = edit_steps * item["n_avg"] * calls_per_edit_step
    rows.append({
        "exp_name": item["exp_name"],
        "solver_type": item["solver_type"],
        "T_steps": item["T_steps"],
        "n_max": item["n_max"],
        "n_avg": item["n_avg"],
        "estimated_nfe": estimated_nfe,
        "pc_lambda": item.get("pc_guidance_lambda", ""),
        "pc_gamma": item.get("pc_guidance_gamma", ""),
        "pc_weight": item.get("pc_guidance_weight", ""),
    })

display(pd.DataFrame(rows))
print(Path(EXP_YAML).read_text())


In [ ]:
# 4) Run original FlowEdit vs CFG++ no-first-term variants under the same estimated NFE
import shutil
import subprocess
from pathlib import Path
import pandas as pd

cleanup_paths = [
    "outputs/SameNFE_OriginalFlowEdit_Euler18",
    "outputs/SameNFE_CFGPPNoFirstTerm_L050_W100",
    "outputs/SameNFE_CFGPPNoFirstTerm_L100_W100",
    "outputs/SameNFE_CFGPPNoFirstTerm_L150_W100",
    "outputs/run_summary.csv",
    QUALITY_PER_SAMPLE_CSV,
    QUALITY_SUMMARY_CSV,
    ARTIFACT_PER_SAMPLE_CSV,
    ARTIFACT_SUMMARY_CSV,
    COMPARISON_CSV,
]
for path in cleanup_paths:
    p = Path(path)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)
subprocess.run(["python", "run_script.py", "--device_number", "0", "--exp_yaml", EXP_YAML], check=True)

print("Run summary:")
display(pd.read_csv("outputs/run_summary.csv"))


In [ ]:
# 5) Evaluate generation quality and artifact proxy
import subprocess
from pathlib import Path
import pandas as pd

subprocess.run([
    "python", "evaluate_clip_dino.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--dataset_yaml", DATASET_YAML,
    "--out_samples", QUALITY_PER_SAMPLE_CSV,
    "--out_summary", QUALITY_SUMMARY_CSV,
], check=True)

subprocess.run([
    "python", "evaluate_artifact_proxy.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--out_samples", ARTIFACT_PER_SAMPLE_CSV,
    "--out_summary", ARTIFACT_SUMMARY_CSV,
], check=True)

quality = pd.read_csv(QUALITY_SUMMARY_CSV)
artifact = pd.read_csv(ARTIFACT_SUMMARY_CSV)
summary = quality.merge(
    artifact,
    on=[
        "exp_name",
        "solver_type",
        "estimated_nfe",
        "pc_guidance_lambda",
        "pc_guidance_gamma",
        "pc_guidance_weight",
        "num_samples",
    ],
    how="left",
)
display(summary)
summary.to_csv(COMPARISON_CSV, index=False)
print("Wrote:", COMPARISON_CSV)


In [ ]:
# 6) Per-sample same-NFE comparison table
from pathlib import Path
import pandas as pd

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
samples = quality_samples.merge(
    artifact_samples[[
        "exp_name", "solver_type", "source_image", "target_index",
        "edited_artifact_proxy", "artifact_proxy_delta_vs_source",
        "edited_clipping_ratio", "edited_high_saturation_ratio",
    ]],
    on=["exp_name", "solver_type", "source_image", "target_index"],
    how="left",
)
samples["case"] = samples["source_image"].map(lambda p: Path(p).stem)

metrics = [
    "clip_alignment",
    "dino_similarity",
    "edit_preservation_score",
    "edited_artifact_proxy",
    "artifact_proxy_delta_vs_source",
    "elapsed_seconds",
]
for col in metrics:
    samples[col] = samples[col].astype(float)

columns = [
    "case",
    "exp_name",
    "solver_type",
    "estimated_nfe",
    "pc_guidance_weight",
] + metrics
display(samples[columns].sort_values(["case", "exp_name"]))


In [ ]:
# 7) Image comparison table for visual artifact inspection
import base64
import html
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import HTML, display

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
samples = quality_samples.merge(
    artifact_samples[[
        "exp_name", "solver_type", "source_image", "target_index",
        "edited_artifact_proxy", "artifact_proxy_delta_vs_source",
    ]],
    on=["exp_name", "solver_type", "source_image", "target_index"],
    how="left",
)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

exp_columns = [
    ("SameNFE_OriginalFlowEdit_Euler18", "Original FlowEdit Euler"),
    ("SameNFE_CFGPPNoFirstTerm_L050_W100", "No first term ?=0.50"),
    ("SameNFE_CFGPPNoFirstTerm_L100_W100", "No first term ?=1.00"),
    ("SameNFE_CFGPPNoFirstTerm_L150_W100", "No first term ?=1.50"),
]

def image_to_data_uri(path):
    path = Path(path)
    suffix = path.suffix.lower().replace(".", "")
    if suffix == "jpg":
        suffix = "jpeg"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/{suffix};base64,{data}"

def img_tag(path, width=220):
    return f"<img src='{image_to_data_uri(path)}' style='width:{width}px;max-width:100%;border:1px solid #ddd;'>"

def metric_line(row):
    return (
        f"CLIP {float(row['clip_alignment']):.4f} / "
        f"DINO {float(row['dino_similarity']):.4f} / "
        f"Artifact {float(row['edited_artifact_proxy']):.4f}"
    )

rows = []
for item in dataset:
    source = item["input_img"]
    subset = samples[samples["source_image"] == source]
    cells = [f"<td><b>{html.escape(Path(source).stem)}</b><br>{img_tag(source)}</td>"]
    for exp_name, label in exp_columns:
        exp_subset = subset[subset["exp_name"] == exp_name]
        if exp_subset.empty:
            cells.append(f"<td><b>{html.escape(label)}</b><br><em>missing output</em></td>")
            continue
        row = exp_subset.iloc[0]
        cells.append(
            f"<td><b>{html.escape(label)}</b><br>{img_tag(row['output_image'])}<br>{metric_line(row)}</td>"
        )
    rows.append("<tr>" + "".join(cells) + "</tr>")

headers = "".join(f"<th>{html.escape(label)}</th>" for _, label in exp_columns)
table_html = (
    "<table style='border-collapse:collapse;width:100%;'>\n"
    f"<thead><tr><th>Source</th>{headers}</tr></thead>\n"
    "<tbody>\n"
    + "\n".join(rows)
    + "\n</tbody></table>"
)
display(HTML(table_html))


## How to Interpret

- Same NFE means the comparison controls for the number of neural-network flow evaluations.
- This branch tests the literal remove-CFG++-first-term idea: only scheduled midpoint contrast remains.
- Higher CLIP alignment means better target-prompt matching.
- Higher DINO similarity means stronger source structure preservation.
- Lower artifact proxy and lower artifact delta suggest fewer saturation/clipping artifacts, but this is only a lightweight proxy; use the image table for final qualitative judgment.
- Because the current anchor term is removed, `lambda=0.5` may be too weak; `lambda=1.0` and `lambda=1.5` test whether stronger alpha can compensate.
- A convincing result would improve target alignment without the visual outputs becoming under-edited or unstable.
